# loralake on Colab

运行时类型选 **L4 GPU** 或更好的（A100 / G4 / H100）。T4 不支持 bf16，不适合训练。

Google Drive 的 `MyDrive/loralake/` 用来长期保存，会话结束后不会丢：

```
MyDrive/loralake/
├── presets/               预设 *.yaml，会覆盖仓库里的同名预设；default.yaml 由下面的「预设参数」单元格生成
├── models/                ComfyUI 格式的 models 目录
│   ├── diffusion_models/  anima-base*.safetensors
│   ├── text_encoders/     qwen_3_06b*.safetensors
│   └── vae/               qwen_image_vae*.safetensors
├── datasets/              数据集
├── output/                训练好的 LoRA
└── .cache/                HF 镜像索引
```

启动时把 Drive 上已有的内容复制到本地 `/content/work`，之后下载、预处理、训练全部在本地读写。
下载结束后把新增的数据集同步回 Drive，训练结束后把本次的输出目录（LoRA、`run.json`、`train.log` 等）移动到 Drive 的 `output/`；
也可以随时运行最后的「保存到 Drive」单元格。**会话中途断开时，还没保存的内容会丢失。**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/loralake"
LOCAL = "/content/work"
REPO = "/content/loralake"

## 安装

每个新会话都要运行一次，约几分钟。

In [ ]:
import os
if not os.path.isdir(REPO):
    !git clone --recurse-submodules https://github.com/Zhou-London/loralake.git {REPO}
else:
    !git -C {REPO} checkout -- presets && git -C {REPO} pull --recurse-submodules
!cd {REPO} && ./install.sh

## 从 Drive 复制到本地

只复制本地没有或有变化的文件。

In [ ]:
import pathlib, shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def sync(src, dst):
    """把 src 下本地没有、大小不同或更新过的文件复制到 dst（不删除 dst 中多出的文件）。"""
    src, dst = Path(src), Path(dst)
    if not src.is_dir():
        return
    jobs = []
    for file in src.rglob("*"):
        if not file.is_file():
            continue
        target = dst / file.relative_to(src)
        stat = file.stat()
        if target.exists():
            old = target.stat()
            if old.st_size == stat.st_size and int(old.st_mtime) >= int(stat.st_mtime):
                continue
        jobs.append((file, target, stat.st_size))
    def copy(job):
        file, target, _ = job
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file, target)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(copy, jobs))
    size = sum(job[2] for job in jobs) / 2**30
    print(f"{src} → {dst}: {len(jobs)} 个文件, {size:.2f} GB")

def pull():
    sync(f"{DRIVE}/models", f"{LOCAL}/models")
    sync(f"{DRIVE}/datasets", f"{LOCAL}/datasets")
    sync(f"{DRIVE}/presets", f"{REPO}/presets")
    sync(f"{DRIVE}/.cache", f"{REPO}/.cache")

def move_output():
    """把本地 output/ 下的训练目录移动到 Drive 的 output/，本地不再保留。"""
    local = Path(LOCAL, "output")
    if not local.is_dir():
        return
    for run in sorted(p for p in local.iterdir() if p.is_dir()):
        target = Path(DRIVE, "output", run.name)
        # 续训后本地的训练状态已更新或删除，Drive 上旧的那份不再需要
        for state in [*target.glob("checkpoints/*/global_step*"), *target.glob("checkpoints/*/latest")]:
            if not (run / state.relative_to(target)).exists():
                shutil.rmtree(state) if state.is_dir() else state.unlink()
        sync(run, target)
        shutil.rmtree(run)
        for lora in sorted(target.glob("*.safetensors")):
            print(f"  {lora}")

def pull_run(name):
    """把 Drive 上中断的训练输出复制回本地，用于续训。"""
    sync(f"{DRIVE}/output/{name}", f"{LOCAL}/output/{name}")

def push():
    sync(f"{LOCAL}/datasets", f"{DRIVE}/datasets")
    sync(f"{REPO}/.cache", f"{DRIVE}/.cache")
    move_output()

for sub in ("models/diffusion_models", "models/text_encoders", "models/vae", "presets", "datasets", "output"):
    Path(DRIVE, sub).mkdir(parents=True, exist_ok=True)

pull()


## 配置

填写变量

In [ ]:
CONFIG = """
DANBOORU_LOGIN=
DANBOORU_API_KEY=
HF_API_KEY=
PROXY=
DOWNLOAD_WORKERS=8
MIRROR_WORKERS=32
"""

`presets/default.yaml`：不加 `-c` 时使用这个预设，命令行参数优先。

In [ ]:
PRESET = """
preprocess:
  trigger: ""
  exclude_images:
    - ai-generated
    - ai-assisted
    - lowres
    - jpeg_artifacts
    - bad_anatomy
    - bad_hands
    - bad_feet
    - bad_proportions
    - bad_perspective
    - anatomical_nonsense
    - sketch
    - unfinished
  remove_tags: []
  prefix_tags: []
  include_rating: true
  include_year: true
  include_artist: true

train:
  rank: 32
  learning_rate: 2.0e-5
  epochs: 10
  batch_size: 4
  resolution: 1024
  save_every_epochs: 2
  checkpoint_every_minutes: 30
  optimizer: adamw_optimi
  caption_dropout: 0
  gradient_checkpointing: true
"""

In [ ]:
# 应用：写入 config 和 presets/default.yaml（预设同时保存到 Drive）
Path(REPO, "config").write_text(CONFIG.lstrip() + f"""
COMFY_MODELS_PATH={LOCAL}/models
DATASET_DIR={LOCAL}/datasets
OUTPUT_DIR={LOCAL}/output
DERIVED_DIR={LOCAL}/derived
""", encoding="utf-8")
for path in (Path(REPO, "presets", "default.yaml"), Path(DRIVE, "presets", "default.yaml")):
    path.write_text(PRESET.lstrip(), encoding="utf-8")

os.chdir(REPO)
# ! 若输出错乱可删掉这三行
os.environ["TTY_COMPATIBLE"] = "1"
os.environ["TTY_INTERACTIVE"] = "1"
os.environ["COLUMNS"] = "120"

## 使用

按需修改命令。参数说明见 [README](https://github.com/Zhou-London/loralake#readme)。加 `-y` 可跳过确认；用其他预设时加 `-c 名称`，例如 Drive 上的 `presets/miku.yaml` 对应 `-c miku`。

**下载或训练结束后运行最后的「保存到 Drive」单元格。**

训练可以随时中断（停止单元格），然后运行「保存到 Drive」。训练状态每 30 分钟保存一次，之后（包括新会话）用下面的续训单元格从最近一次保存的状态继续，名称就是 `output/` 下的目录名。

In [ ]:
!llk -s 初音未来

In [ ]:
!llk -d hatsune_miku --limit 300

In [ ]:
!llk -p hatsune_miku

In [ ]:
!llk -t hatsune_miku

续训：把 Drive 上中断的训练输出复制回本地后继续。

In [ ]:
RUN = "hatsune_miku_20261124-1030"
pull_run(RUN)
!llk --resume {RUN}

## 保存到 Drive

把新增的数据集和 HF 索引同步到 Drive，并把本地的训练输出移动到 Drive 的 `output/`。会话断开前没运行的话，这些内容会丢失。

In [ ]:
push()